# Import libraries

In [1]:
%load_ext autoreload
%autoreload 2
import os
import sys
import json
import glob
from collections import defaultdict

os.environ["CUDA_VISIBLE_DEVICES"] = "5"

from tqdm import tqdm

from filtering import (
    get_args,
    load_model,
    output_to_jsonl,
    filtering,
    load_real_data,
    load_syn_data,
    compute_average_grads,
    calculate_recon_loss_ids,
)

# Parameters

In [10]:
json_file = 'synthetic_data'
file_dir = '/data/yutong/synthetic_data/'
exp_pattern = 'sst2-validation-phi-nreal50-steps30-nsyn10-admm'

_all = glob.glob(os.path.join(file_dir, f'{exp_pattern}*', 'SST2*'))
training_dir = [d for d in _all if os.path.exists(os.path.join(d, f'{json_file}.jsonl'))]
print(f'Found {len(training_dir)} valid run dirs (out of {len(_all)} matched)')
for run in sorted(training_dir):
    print(f'  {run}')

input_flags = [sys.argv[0],
               '--dataset', 'sst2',
               '--model_name', 'phi',
               '--pos_label', 'positive',
               '--neg_label', 'negative',
               '--gen_bs', '10',
               '--use_instruction', 'false',
               '--use_fewshot', 'true',
               '--filter_score', 'cls',
               '--filter_method', 'remove',
               '--coeff_perplexity', '0',
               '--top_n', '50',
               '--file_dir', file_dir,
               '--json_file', json_file,
               '--clean', 'true',
               '--balance_score', 'false',
               '--per_label', 'true',
               '--interleave_label', 'false',
]
sys.argv = input_flags

args = get_args()
for arg in vars(args):
    print(f"{arg}: {getattr(args, arg)}")


Found 12 valid run dirs (out of 20 matched)
  /data/yutong/synthetic_data/sst2-validation-phi-nreal50-steps30-nsyn10-admm-dp_eps0.05-rho0.001-inner50-seed42_2026-05-21_03-10-27/SST2-validation-phi-nreal100-steps30-nsyn10-admm-dp_eps0.05-rho0.001-inner50-seed42
  /data/yutong/synthetic_data/sst2-validation-phi-nreal50-steps30-nsyn10-admm-dp_eps0.05-rho0.01-inner50-seed42_2026-05-18_15-33-22/SST2-validation-phi-nreal100-steps30-nsyn10-admm-dp_eps0.05-rho0.01-inner50-seed42
  /data/yutong/synthetic_data/sst2-validation-phi-nreal50-steps30-nsyn10-admm-dp_eps0.05-rho0.1-inner50-seed42_2026-05-18_15-33-22/SST2-validation-phi-nreal100-steps30-nsyn10-admm-dp_eps0.05-rho0.1-inner50-seed42
  /data/yutong/synthetic_data/sst2-validation-phi-nreal50-steps30-nsyn10-admm-dp_eps0.05-rho0.5-inner50-seed42_2026-05-18_15-33-22/SST2-validation-phi-nreal100-steps30-nsyn10-admm-dp_eps0.05-rho0.5-inner50-seed42
  /data/yutong/synthetic_data/sst2-validation-phi-nreal50-steps30-nsyn10-admm-dp_eps0.05-rho1-inne

# Load model

In [11]:
tokenizer, model, device = load_model(args.model_name)

# Set last layer gradient
LAST_LAYERS = ["lm_head"] 

named_parameters_to_optim = []

for name, param in model.named_parameters():
    if any(substring in name for substring in LAST_LAYERS):
        named_parameters_to_optim.append((name, param))
    else:
        param.requires_grad = False

assert len(named_parameters_to_optim) != 0, "no layer found"
print(f"Set gradients for {len(named_parameters_to_optim)} layers")

Set gradients for 2 layers


# Filtering - Clean remove

In [ ]:
# Set file_path
args.filter_method = 'remove'
args.gen_bs = 10 # Make sure it matches the setting

# Set real_id
for run in tqdm(training_dir):
    file_path = os.path.join(run, f'{args.json_file}.jsonl')
    samples = []
    
    with open(file_path, 'r') as f:
        for line in f:
            samples.append(json.loads(line))
    
    with open(file_path, 'w') as f:
        for sample in samples:
            sample["real_id"] = sample["id"] // args.gen_bs
            f.write(json.dumps(sample) + "\n")

# Clean remove
filtering(
    args,
    training_dir,
    model,
    tokenizer,
    device
)

  0%|          | 0/12 [00:00<?, ?it/s]

100%|██████████| 200/200 [00:30<00:00,  6.56it/s]
1it [00:30, 30.52s/it]

Mismatch count = 93


100%|██████████| 20/20 [00:03<00:00,  6.63it/s]
2it [00:33, 14.35s/it]

Mismatch count = 11


...

Mismatch count = 101


100%|██████████| 200/200 [00:29<00:00,  6.71it/s]
4it [01:33, 24.78s/it]

Mismatch count = 99


100%|██████████| 200/200 [00:29<00:00,  6.81it/s]
5it [02:02, 26.45s/it]

Mismatch count = 93


# (Re)calculate rec_loss_ids per sample

In [ ]:
# Load real data & compute grad
pos_sequences, neg_sequences, pos_labels, neg_labels = load_real_data(
    dataset_name='sst2',
    split='validation',
    device=device,
    n_gen_samples=100,
    n_fewshot=0,
    random_seed=42,
    subset=50,
)

print(pos_sequences[:5])

real_pos_grads = compute_average_grads(
    args,
    model,
    tokenizer,
    pos_sequences,
    pos_labels
)

real_neg_grads = compute_average_grads(
    args,
    model,
    tokenizer,
    neg_sequences,
    neg_labels
)

['( næs ) directed the stage version of elling , and gets fine performances from his two leads who originated the characters on stage . ', 'you will emerge with a clearer view of how the gears of justice grind on and the death report comes to share airtime alongside the farm report . ', '... a fun little timewaster , helped especially by the cool presence of jean reno . ', 'the weight of the piece , the unerring professionalism of the chilly production , and the fascination embedded in the lurid topic prove recommendation enough . ', "audrey tatou has a knack for picking roles that magnify her outrageous charm , and in this literate french comedy , she 's as morning-glory exuberant as she was in amélie . "]


100%|██████████| 21/21 [00:01<00:00, 17.30it/s]


In [ ]:
if args.dataset in ['sst2', 'rotten_tomatoes']:
    POS_LABEL = 'great'
    NEG_LABEL = 'bad'
elif args.dataset == 'TwitterEmotion':
    POS_LABEL = 'joy'
    NEG_LABEL = 'sadness'

for run in tqdm(training_dir):
    syn_data_path = os.path.join(run, f'synthetic_data_clean_remove_cls_phi_{args.dataset}_{args.pos_label}_{args.neg_label}_instrFalse_fsTrue.jsonl')
    if not os.path.exists(syn_data_path):
        print(syn_data_path)
        continue

    # Load synthetic data
    syn_pos_sequences, syn_neg_sequences = load_syn_data(str(syn_data_path), args.dataset)

    list_raw_pos_loss = calculate_recon_loss_ids(
        syn_pos_sequences,
        [POS_LABEL for _ in range(len(syn_pos_sequences))],
        real_pos_grads,
        model,
        tokenizer,
        dataset=args.dataset
    )

    list_raw_neg_loss = calculate_recon_loss_ids(
        syn_neg_sequences,
        [NEG_LABEL for _ in range(len(syn_neg_sequences))],
        real_neg_grads,
        model,
        tokenizer,
        dataset=args.dataset
    )

    # Read the JSONL file
    samples = []
    with open(syn_data_path, 'r') as file:
        for line in file:
            sample = json.loads(line)
            samples.append(sample)

    # Group samples by label
    grouped_samples = defaultdict(list)
    for sample in samples:
        grouped_samples[sample['label']].append(sample)

    loss_dict = {
        1: list_raw_pos_loss,
        0: list_raw_neg_loss
    }
    list_out_samples = []

    for label, label_samples in grouped_samples.items():
        for sample_idx, sample in enumerate(label_samples):
            sample['rec_loss_ids'] = loss_dict[label][sample_idx]
            list_out_samples.append(sample)

    output_to_jsonl(args, list_out_samples, syn_data_path, post_processing=False)

  0%|          | 0/10 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Pos: 92. Neg: 15


 10%|█         | 1/10 [00:06<00:58,  6.53s/it]

Generating train split: 0 examples [00:00, ? examples/s]

...

Pos: 88. Neg: 15


 90%|█████████ | 9/10 [01:08<00:07,  7.43s/it]

Generating train split: 0 examples [00:00, ? examples/s]

Pos: 87. Neg: 13


100%|██████████| 10/10 [01:16<00:00,  7.68s/it]


# Extract top *score*

In [ ]:
args.filter_method = 'top_score'
args.coeff_perplexity = 0
args.json_file = f'synthetic_data_clean_remove_cls_phi_{args.dataset}_{args.pos_label}_{args.neg_label}_instrFalse_fsTrue'
args.top_n = 20 # Modify here for different budget
args.balance_score = False

filtering(
    args,
    training_dir,
    model,
    tokenizer,
    device,
    num_out=5 if args.top_n == 3 else None
)

10it [00:00, 190.85it/s]

92
15
91
8
88
13
90
17
88
19
87
11
93
10
91
12
88
15
87
13


In [ ]:
args.coeff_perplexity = 0.05

filtering(
    args,
    training_dir,
    model,
    tokenizer,
    device,
    num_out=5 if args.top_n == 3 else None
)

10it [00:00, 254.04it/s]

92
15
Start balancing
Label 0 has a higher average score of 0.3187912873427073 compared to 0.2559668460488319
Finish balancing
91
8
Start balancing
Label 0 has a higher average score of 0.2877886697649956 compared to 0.2516775706410408
Finish balancing
88
13
Start balancing
Label 0 has a higher average score of 0.311844592828017 compared to 0.2516406226158142
Finish balancing
90
17
Start balancing
Label 0 has a higher average score of 0.3325202026787926 compared to 0.2558428066968918
Finish balancing
88
19
Start balancing
Label 0 has a higher average score of 0.3993542997460616 compared to 0.2635898444056511
Finish balancing
87
11
Start balancing
Label 0 has a higher average score of 0.302529359405691 compared to 0.2517813110351563
Finish balancing
93
10
Start balancing
Label 0 has a higher average score of 0.3116655057668686 compared to 0.2568677482008935
Finish balancing
91
12
Start balancing
Label 0 has a higher average score of 0.3045861467719078 compared to 0.25602492332458493
Fin